In [1]:
import os
import pandas as pd
import unicodedata

# === Config Paths ===
base_dir = "../training-data/training-set/"
all_files = [f for f in os.listdir(base_dir) if f.startswith("training_set_") and f.endswith(".csv")]

def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

def wl_str_to_pct(wl_str):
    if isinstance(wl_str, str) and '-' in wl_str:
        try:
            wins, losses = map(int, wl_str.split('-'))
            total = wins + losses
            return wins / total if total > 0 else None
        except:
            return None
    return None

def convert_streak(s):
    if isinstance(s, str) and len(s) > 1:
        direction = 1 if s[0].upper() == 'W' else -1 if s[0].upper() == 'L' else 0
        try:
            count = int(s[1:])
            return direction * count
        except ValueError:
            return None
    return None

def safe_divide(numerator, denominator):
    try:
        num = float(numerator)
        denom = float(denominator)
        return num / denom if denom > 0 else None
    except (ValueError, TypeError):
        return None

# === Loop over each training set file ===
for filename in all_files:
    print(f"🔄 Updating: {filename}")
    path = os.path.join(base_dir, filename)
    df = pd.read_csv(path)

    # Normalize expected numeric columns
    num_cols = ['fav_2B', 'dog_2B', 'fav_3B', 'dog_3B',
                'fav_HR', 'dog_HR', 'fav_RBI', 'dog_RBI',
                'fav_AB', 'dog_AB', 'fav_pitcher_so', 'dog_pitcher_so',
                'fav_pitcher_ip', 'dog_pitcher_ip']
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # === Compute SO/IP
    df['fav_pitcher_so_in'] = df.apply(lambda r: safe_divide(r.get('fav_pitcher_so'), r.get('fav_pitcher_ip')), axis=1)
    df['dog_pitcher_so_in'] = df.apply(lambda r: safe_divide(r.get('dog_pitcher_so'), r.get('dog_pitcher_ip')), axis=1)

    # === Convert Streak
    if 'fav_Strk' in df.columns:
        df['fav_Strk'] = df['fav_Strk'].apply(convert_streak)
    if 'dog_Strk' in df.columns:
        df['dog_Strk'] = df['dog_Strk'].apply(convert_streak)

    # === Convert pythWL
    if 'fav_pythWL' in df.columns:
        df['fav_pythWL_pct'] = df['fav_pythWL'].apply(wl_str_to_pct)
    if 'dog_pythWL' in df.columns:
        df['dog_pythWL_pct'] = df['dog_pythWL'].apply(wl_str_to_pct)

    # === Convert last10/20/30
    for span in ['last10', 'last20', 'last30']:
        fav_col = f'fav_{span}'
        dog_col = f'dog_{span}'
        if fav_col in df.columns:
            df[f'{fav_col}_pct'] = df[fav_col].apply(wl_str_to_pct)
        if dog_col in df.columns:
            df[f'{dog_col}_pct'] = df[dog_col].apply(wl_str_to_pct)

    # === Convert batting stats to per-AB %
    conversion_pairs = [
        ('fav_2B', 'fav_AB', 'fav_2B_pct'),
        ('dog_2B', 'dog_AB', 'dog_2B_pct'),
        ('fav_3B', 'fav_AB', 'fav_3B_pct'),
        ('dog_3B', 'dog_AB', 'dog_3B_pct'),
        ('fav_HR', 'fav_AB', 'fav_HR_pct'),
        ('dog_HR', 'dog_AB', 'dog_HR_pct'),
        ('fav_RBI', 'fav_AB', 'fav_RBI_pct'),
        ('dog_RBI', 'dog_AB', 'dog_RBI_pct'),
    ]
    for stat_col, ab_col, pct_col in conversion_pairs:
        if stat_col in df.columns and ab_col in df.columns:
            df[pct_col] = df.apply(lambda row: safe_divide(row[stat_col], row[ab_col]), axis=1)

    # === Save updated version in place ===
    df.to_csv(path, index=False)
    print(f"✅ Saved enriched: {filename}")


🔄 Updating: training_set_2025-05-31.csv
✅ Saved enriched: training_set_2025-05-31.csv
🔄 Updating: training_set_2025-06-10.csv
✅ Saved enriched: training_set_2025-06-10.csv
🔄 Updating: training_set_2025-05-29.csv
✅ Saved enriched: training_set_2025-05-29.csv
🔄 Updating: training_set_2025-06-01.csv
✅ Saved enriched: training_set_2025-06-01.csv
🔄 Updating: training_set_2025-06-08.csv
✅ Saved enriched: training_set_2025-06-08.csv
🔄 Updating: training_set_2025-05-30.csv
✅ Saved enriched: training_set_2025-05-30.csv
🔄 Updating: training_set_2025-06-11.csv
✅ Saved enriched: training_set_2025-06-11.csv
🔄 Updating: training_set_2025-06-03.csv
✅ Saved enriched: training_set_2025-06-03.csv
🔄 Updating: training_set_2025-06-02.csv
✅ Saved enriched: training_set_2025-06-02.csv
🔄 Updating: training_set_2025-06-07.csv
✅ Saved enriched: training_set_2025-06-07.csv
🔄 Updating: training_set_2025-06-04.csv
✅ Saved enriched: training_set_2025-06-04.csv
🔄 Updating: training_set_2025-06-05.csv
✅ Saved enrich